# Prediksi Risiko Piutang Transaksi - PT. Groovy Layan Satwa
**Kelompok 12 - Pembelajaran Mesin (S1) - Universitas Nusa Mandiri - Gasal 2026/2027**

Notebook ini merangkum Tugas 02-06: pembersihan data, baseline 3 algoritma, GridSearchCV, Permutation Importance & SHAP,
serialisasi model, Optuna + Stratified 5-Fold CV, dan kalibrasi decision threshold.

**Cara pakai di Google Colab:** jalankan sel instalasi, unggah `detil_penjualan_2026_gabungan.csv`, lalu *Runtime > Run all*.

In [ ]:
!pip -q install optuna shap

In [ ]:
import os, json, warnings
warnings.filterwarnings('ignore')
try:
    from google.colab import files
    if not os.path.exists('detil_penjualan_2026_gabungan.csv'):
        files.upload()                      # pilih detil_penjualan_2026_gabungan.csv
except ImportError:
    pass
PATH = 'detil_penjualan_2026_gabungan.csv'
RS = 42

## Tugas 02 - Pembersihan data, rekayasa fitur & preprocessor

In [ ]:
# Modul bersama Kelompok 12: pembersihan data Minggu 02 & preprocessor Minggu 03
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler, StandardScaler

def load_clean(path):
    df = pd.read_csv(path, sep=';')
    fmt = '%d-%m-%Y %H:%M:%S'
    t_order = pd.to_datetime(df['Waktu Order'], format=fmt)
    t_bayar = pd.to_datetime(df['Waktu Bayar'], format=fmt)
    metode = df['Metode Pembayaran']
    out = pd.DataFrame({
        'Layanan': (metode.str.split(',').str[0]
                    .str.replace(r'^(Bank Transfer|Kartu Debit/Kredit) - ', '', regex=True)
                    .str.replace(r'[- ](Piutang|Trans|QR|Card|Cash|Tokopedia)$', '', regex=True)
                    .str.replace('Day Care', 'Daycare').str.strip()),
        'Jenis_Order': df['Jenis Order'],
        'Total_Penjualan': df['Total Penjualan (Rp)'],
        'Jumlah_Item': df['Produk'].fillna(df['Produk'].mode()[0]).str.count(r',(?!\s?\d)') + 1,
        'Jam_Order': t_order.dt.hour,
        'Hari_Order': t_order.dt.dayofweek,
        'Bulan_Order': t_order.dt.month,
        'Durasi_Bayar_Menit': (t_bayar - t_order).dt.total_seconds() / 60,
        'Status_Piutang': metode.str.contains('Piutang').astype(int),
    })
    out.loc[metode.eq('Cash'), 'Layanan'] = 'Sales'
    return out

FITUR_KAT = ['Layanan', 'Jenis_Order']
FITUR_ORD = ['Hari_Order', 'Bulan_Order']
FITUR_ROBUST = ['Total_Penjualan', 'Jumlah_Item']
FITUR_STD = ['Jam_Order']
FITUR = FITUR_KAT + FITUR_ORD + FITUR_ROBUST + FITUR_STD
class Winsorizer(BaseEstimator, TransformerMixin):
    """Memotong nilai di atas persentil q yang dipelajari dari data latih."""
    def __init__(self, q=0.99):
        self.q = q
    def fit(self, X, y=None):
        self.cap_ = np.nanquantile(np.asarray(X, dtype=float), self.q, axis=0)
        return self
    def transform(self, X):
        return np.minimum(np.asarray(X, dtype=float), self.cap_)
    def get_feature_names_out(self, input_features=None):
        return np.asarray(input_features, dtype=object)

def make_preprocessor():
    return ColumnTransformer([
        ('ohe', OneHotEncoder(handle_unknown='ignore'), FITUR_KAT),
        ('ord', 'passthrough', FITUR_ORD),
        ('robust', Pipeline([('wins', Winsorizer(0.99)), ('sc', RobustScaler())]), FITUR_ROBUST),
        ('std', StandardScaler(), FITUR_STD),
    ])

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, cross_validate, cross_val_predict
df = load_clean(PATH)
X, y = df[FITUR], df['Status_Piutang']
print(df.shape); print(y.value_counts(normalize=True).round(3))
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RS, stratify=y)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RS)
print('Train', X_train.shape, 'Test', X_test.shape)

## Tugas 03 - Baseline 3 algoritma + 5-Fold CV

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score, roc_auc_score, average_precision_score
def pipe(model): return Pipeline([('prep', make_preprocessor()), ('model', model)])
baseline = {
    'Logistic Regression': pipe(LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RS)),
    'Decision Tree': pipe(DecisionTreeClassifier(max_depth=6, min_samples_leaf=20, class_weight='balanced', random_state=RS)),
    'Random Forest': pipe(RandomForestClassifier(n_estimators=300, max_depth=12, min_samples_leaf=5, class_weight='balanced', n_jobs=-1, random_state=RS)),
}
for n, p in baseline.items():
    s = cross_val_score(p, X_train, y_train, cv=cv, scoring='f1_macro')
    p.fit(X_train, y_train); pr = p.predict_proba(X_test)[:, 1]
    print(f'{n:20s} CV F1-Macro {s.mean():.4f}±{s.std():.4f} | Test F1-Macro {f1_score(y_test, pr >= .5, average="macro"):.4f} | AUC {roc_auc_score(y_test, pr):.4f}')

## Tugas 04 - GridSearchCV, Permutation Importance & SHAP

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.inspection import permutation_importance
grid = GridSearchCV(baseline['Random Forest'], {'model__n_estimators': [300, 500], 'model__max_depth': [8, 12, 16, None],
                    'model__min_samples_leaf': [2, 5, 10], 'model__max_features': ['sqrt', 0.5]},
                    cv=cv, scoring='f1_macro', n_jobs=-1).fit(X_train, y_train)
tuned_t04 = grid.best_estimator_
print(grid.best_params_, round(grid.best_score_, 4))
pi = permutation_importance(tuned_t04, X_test, y_test, scoring='f1_macro', n_repeats=30, random_state=RS, n_jobs=-1)
print(pd.Series(pi.importances_mean, FITUR).sort_values(ascending=False).round(4))

In [ ]:
import shap
prep, rf = tuned_t04.named_steps['prep'], tuned_t04.named_steps['model']
Z_test = pd.DataFrame(prep.transform(X_test), columns=prep.get_feature_names_out())
sv = shap.TreeExplainer(rf)(Z_test)[:, :, 1]
shap.plots.beeswarm(sv, max_display=12)
shap.plots.waterfall(sv[list(X_test.index).index(149)], max_display=12)

## Tugas 05 - Serialisasi pipeline utuh

In [ ]:
import joblib, hashlib, sklearn
joblib.dump(tuned_t04, 'model_final_terbaik.joblib', compress=3)
meta = {'file': 'model_final_terbaik.joblib', 'fitur_input': FITUR, 'scikit_learn': sklearn.__version__,
        'sha256': hashlib.sha256(open('model_final_terbaik.joblib', 'rb').read()).hexdigest()}
json.dump(meta, open('model_metadata.json', 'w'), indent=2)
loaded = joblib.load('model_final_terbaik.joblib')
contoh = pd.DataFrame([{'Layanan': 'Delivery', 'Jenis_Order': 'Lainnya', 'Hari_Order': 1, 'Bulan_Order': 6,
                        'Total_Penjualan': 1600000, 'Jumlah_Item': 1, 'Jam_Order': 14}])[FITUR]
print(loaded.predict(contoh), loaded.predict_proba(contoh).round(4))

## Tugas 06 - Optuna (TPE) + Stratified 5-Fold CV untuk 2 keluarga model

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
def obj_rf(trial):
    p = dict(n_estimators=trial.suggest_int('n_estimators', 50, 400), max_depth=trial.suggest_int('max_depth', 4, 30),
             min_samples_split=trial.suggest_int('min_samples_split', 2, 30), min_samples_leaf=trial.suggest_int('min_samples_leaf', 1, 20),
             max_features=trial.suggest_float('max_features', 0.15, 1.0),
             class_weight=trial.suggest_categorical('class_weight', ['balanced', 'balanced_subsample']))
    return cross_val_score(pipe(RandomForestClassifier(n_jobs=-1, random_state=RS, **p)), X_train, y_train, cv=cv, scoring='f1_macro').mean()
def obj_hgb(trial):
    p = dict(learning_rate=trial.suggest_float('learning_rate', 0.01, 0.3, log=True), max_iter=trial.suggest_int('max_iter', 50, 500),
             max_depth=trial.suggest_int('max_depth', 3, 12), min_samples_leaf=trial.suggest_int('min_samples_leaf', 5, 120),
             l2_regularization=trial.suggest_float('l2_regularization', 1e-4, 10.0, log=True), max_leaf_nodes=trial.suggest_int('max_leaf_nodes', 8, 64))
    return cross_val_score(pipe(HistGradientBoostingClassifier(class_weight='balanced', random_state=RS, **p)), X_train, y_train, cv=cv, scoring='f1_macro').mean()
studies = {}
for nama, obj in [('RandomForest', obj_rf), ('HistGradientBoosting', obj_hgb)]:
    st = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=RS))
    st.optimize(obj, n_trials=40)
    studies[nama] = st
    st.trials_dataframe().to_csv(f'riwayat_optuna_{nama}.csv', index=False)
    print(nama, round(st.best_value, 4), st.best_params)

In [ ]:
terbaik = max(studies, key=lambda k: studies[k].best_value)
bp = studies[terbaik].best_params
final = pipe(RandomForestClassifier(n_jobs=-1, random_state=RS, **bp)) if terbaik == 'RandomForest' else \
        pipe(HistGradientBoostingClassifier(class_weight='balanced', random_state=RS, **bp))
final.fit(X_train, y_train)
y_probs = final.predict_proba(X_test)[:, 1]
print('Model final:', terbaik)
print(classification_report(y_test, (y_probs >= 0.5).astype(int), target_names=['Lunas (0)', 'Piutang (1)'], digits=4))
print('ROC-AUC', round(roc_auc_score(y_test, y_probs), 4), '| PR-AUC', round(average_precision_score(y_test, y_probs), 4))

### Decision threshold: dipilih dari prediksi out-of-fold data latih (biaya FN = 3 x FP), lalu diuji di Test Set

In [ ]:
oof = cross_val_predict(final, X_train, y_train, cv=cv, method='predict_proba')[:, 1]
grid_t = np.round(np.arange(0.20, 0.81, 0.01), 2)
biaya = [3 * ((oof < t) & (y_train == 1)).sum() + ((oof >= t) & (y_train == 0)).sum() for t in grid_t]
t_opt = grid_t[int(np.argmin(biaya))]; print('Threshold optimal (OOF):', t_opt)
for t in sorted({0.30, 0.40, 0.50, 0.60, 0.70, float(t_opt)}):
    pr = (y_probs >= t).astype(int); tn, fp, fn, tp = confusion_matrix(y_test, pr).ravel()
    print(f't={t:.2f} precision={tp/(tp+fp):.4f} recall={tp/(tp+fn):.4f} F1={f1_score(y_test, pr):.4f} FN={fn} FP={fp}')
joblib.dump({'pipeline': final, 'threshold': float(t_opt)}, 'model_final_tugas06.joblib', compress=3)